# Datas e fusos horários

Use datas para prazos, duração e agrupamento por período. UTC é uma referência; converter para uma cidade muda a representação do mesmo instante. Use America/Sao_Paulo e America/Cuiaba, que consideram regras por data, em vez de subtrair horas manualmente. Em exemplos de outubro de 2026, 15h UTC corresponde a 12h em São Paulo e 11h em Cuiabá.

Execute as demonstrações; depois tente os exercícios antes do gabarito. Escreva dados disponíveis, resultado esperado e passos em português. Requisitos: Python 3.9 ou superior, pandas e IPython. Para fusos, o ambiente precisa de dados IANA; se houver ZoneInfoNotFoundError, instale tzdata no kernel. Não é necessário baixar datasets.

## Roteiro da professora — aula integrada de 60 minutos

Os quatro notebooks pertencem a **uma aula de aproximadamente uma hora no total**. Execute as células necessárias de cima para baixo; a agenda seleciona demonstrações, não todos os exercícios. Cada exercício tem uma resolução logo após a célula de tentativa, com resultado visível quando necessário. Os gabaritos e oficinas originais continuam disponíveis para estudo posterior.

| Tempo | Conteúdo | Material |
|---|---|---|
| 0–5 min | Texto, data e problema de negócio | Datas básico |
| 5–12 min | Ler, formatar e somar datas | Datas básico |
| 12–22 min | Fuso, UTC e mudança de dia | Datas básico |
| 22–27 min | Converter uma coluna e localizar | Datas nível 2 |
| 27–32 min | O que é regex e leitura do padrão | Regex básico |
| 32–42 min | Procurar, validar e limpar | Regex básico |
| 42–49 min | Reutilizar a regra em função | Regex básico |
| 49–55 min | Aplicar padrões a uma coluna | Regex nível 2 |
| 55–60 min | Verificação e fechamento | Perguntas do roteiro |

**Preparação:** abra os quatro arquivos antes da aula e selecione o kernel Python. Os níveis 2 precisam de pandas e IPython; a demonstração com `format="ISO8601"` requer pandas 2.0 ou superior. Teste os imports previamente. A biblioteca `zoneinfo` precisa dos dados IANA; se o ambiente não os encontrar, instale `tzdata` no kernel. Os básicos usam a biblioteca padrão, mas podem precisar desses dados de fusos.

Se houver atraso, priorize os exemplos básicos e mostre apenas as tabelas dos níveis 2, preservando os cinco minutos finais de verificação.


In [2]:
import pandas as pd
from IPython.display import display

### Apoio de pandas — ponte com o básico
Um DataFrame é uma tabela; uma Series é uma coluna. `pedidos["inicio"]` seleciona uma coluna. `pd.to_datetime` interpreta vários textos. `format` declara como lê-los; `errors="coerce"` transforma entradas inválidas em `NaT` (“data ausente”). `31/02/2026` não existe: não é uma duração de zero dias. `.dt` permite acessar propriedades e métodos de datas em uma coluna.

Esta demonstração é **opcional na aula de uma hora**. Use depois para aprofundar prazos.

## Demo 1 Interpretar datas
format informa o padrão; errors="coerce" transforma dados inválidos em NaT. A divisão por dias mede duração, não mudança de data no calendário.

In [9]:
pedidos = pd.DataFrame({"pedido": [1,2,3], "inicio": ["01/10/2026","03/10/2026","31/02/2026"], "fim": ["04/10/2026","08/10/2026","09/10/2026"]})
pedidos

for coluna in ["inicio", "fim"]:
    pedidos[coluna] = pd.to_datetime(pedidos[coluna], format="%d/%m/%Y", errors="coerce")

pedidos["dias"] = (pedidos["fim"]-pedidos["inicio"]).dt.total_seconds()/86400
display(pedidos)

,pedido,inicio,fim,dias
0,1,2026-10-01,2026-10-04,3.0
1,2,2026-10-03,2026-10-08,5.0
2,3,NaT,2026-10-09,NaN


In [10]:
pedidos = pd.DataFrame({"pedido": [1,2,3], "inicio": ["01/10/2026","03/10/2026","31/02/2026"], "fim": ["04/10/2026","08/10/2026","09/10/2026"]})

for coluna in ["inicio", "fim"]:
    pedidos[coluna] = pd.to_datetime(pedidos[coluna], format="%d/%m/%Y", errors="coerce")
    
pedidos["dias"] = (pedidos["fim"]-pedidos["inicio"]).dt.total_seconds()/86400
display(pedidos)

,pedido,inicio,fim,dias
0,1,2026-10-01,2026-10-04,3.0
1,2,2026-10-03,2026-10-08,5.0
2,3,NaT,2026-10-09,NaN


### Demonstração selecionada — 22–25 minutos
“No básico convertemos um horário. Aqui aplicamos a transformação a uma coluna inteira.” `Z` no fim do texto indica UTC. `utc=True` uniformiza a coluna em UTC; neste exemplo a origem é explicitamente UTC. `errors="coerce"` mantém o inválido visível como `NaT`. `.dt.tz_convert(...)` converte cada instante.

Mostre o evento às **03:30 UTC**: São Paulo mostra **00:30 de 05/10**, Cuiabá **23:30 de 04/10**. Pergunte em qual dia cada relatório contaria esse evento. Resposta: depende do fuso definido para o relatório.

## Demo 2 UTC para duas cidades
Z indica UTC. utc=True interpreta textos sem fuso como UTC também: só faça isso se a origem realmente usar UTC. tz_convert preserva o instante. Primeiro converta, depois extraia a data local.

In [11]:
eventos = pd.DataFrame({"id": [1,2,3,4], "texto_utc": ["2026-10-05T15:00:00Z","2026-10-05T02:30:00Z","2026-10-05T03:30:00Z","invalido"]})
eventos

,id,texto_utc
0,1,2026-10-05T15:00:00Z
1,2,2026-10-05T02:30:00Z
2,3,2026-10-05T03:30:00Z
3,4,invalido


In [12]:
eventos = pd.DataFrame({"id": [1,2,3,4], "texto_utc": ["2026-10-05T15:00:00Z","2026-10-05T02:30:00Z","2026-10-05T03:30:00Z","invalido"]})


eventos["utc"] = pd.to_datetime(eventos["texto_utc"], format="ISO8601", utc=True, errors="coerce")
eventos["sp"] = eventos["utc"].dt.tz_convert("America/Sao_Paulo")
eventos["cuiaba"] = eventos["utc"].dt.tz_convert("America/Cuiaba")
display(eventos)


,id,texto_utc,utc,sp,cuiaba
0,1,2026-10-05T15:00:00Z,2026-10-05 15:00:00+00:00,2026-10-05 12:00:00-03:00,2026-10-05 11:00:00-04:00
1,2,2026-10-05T02:30:00Z,2026-10-05 02:30:00+00:00,2026-10-04 23:30:00-03:00,2026-10-04 22:30:00-04:00
2,3,2026-10-05T03:30:00Z,2026-10-05 03:30:00+00:00,2026-10-05 00:30:00-03:00,2026-10-04 23:30:00-04:00
3,4,invalido,NaT,NaT,NaT


### Localizar versus converter — 25–27 minutos
“Recebi `09:00` e a origem confirmou que é horário de Cuiabá. Primeiro preciso registrar essa informação.” `tz_localize` **atribui** o fuso a uma hora sem fuso, mantendo 09:00. Depois `tz_convert` mostra o mesmo instante em outro relógio: **13:00 UTC e 10:00 em São Paulo**.

`ambiguous="raise"` e `nonexistent="raise"` interrompem a operação se uma transição de horário tornar aquela hora ambígua ou inexistente. Não precisamos aprofundar isso hoje. Pergunta: posso usar `utc=True` para declarar que 09:00 de Cuiabá era UTC? **Não**, isso interpretaria outro instante.

## Demo 3 Localizar não é converter
Uma hora sem fuso não informa um instante único. tz_localize atribui o fuso de origem sem mudar o relógio; tz_convert muda a representação para outro fuso. Aqui a origem declarou que o texto é horário de Cuiabá. Datas históricas de transição podem ser ambíguas ou inexistentes; usamos raise para não escolher silenciosamente.

In [13]:
local = pd.to_datetime(pd.Series(["2026-10-05 09:00:00"]), format="%Y-%m-%d %H:%M:%S")
local_cuiaba = local.dt.tz_localize("America/Cuiaba", ambiguous="raise", nonexistent="raise")
display(pd.DataFrame({"cuiaba":local_cuiaba, "utc":local_cuiaba.dt.tz_convert("UTC"), "sp":local_cuiaba.dt.tz_convert("America/Sao_Paulo")}))

,cuiaba,utc,sp
0,2026-10-05 09:00:00-04:00,2026-10-05 13:00:00+00:00,2026-10-05 10:00:00-03:00


## Exercícios

### 1 Prazos
Liste datas inválidas e pedidos que demoraram mais de 4 dias. Não confunda NaT com prazo zero.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [ ]:
# Sua solução aqui.

### Resolução para mostrar na aula

### 1 Prazos

Como pensar: Verificar nulos → selecionar durações válidas acima de 4.

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [ ]:
display(pedidos.loc[pedidos["inicio"].isna() | pedidos["fim"].isna()])
display(pedidos.loc[pedidos["dias"] > 4])
assert pedidos.loc[1,"dias"] == 5

### 2 Mudança de dia
Conte eventos por data local em SP e Cuiabá. Separe registros inválidos. Explique o evento de 03h30 UTC: em SP é dia 5, em Cuiabá ainda é dia 4.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [ ]:
# Sua solução aqui.

### Resolução para mostrar na aula

### 2 Mudança de dia

Como pensar: Separar válidos → converter → extrair data → contar.

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [ ]:
validos = eventos.loc[eventos["utc"].notna()].copy()
for cidade in ["sp","cuiaba"]:
    display(validos[cidade].dt.date.value_counts().sort_index())
display(eventos.loc[eventos["utc"].isna()])
assert eventos.loc[2,"sp"].day == 5
assert eventos.loc[2,"cuiaba"].day == 4

### 3 Janela comercial
Selecione eventos entre 08h inclusive e 18h exclusive em cada cidade. Mostre IDs.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [ ]:
# Sua solução aqui.

### Resolução para mostrar na aula

### 3 Janela comercial

Como pensar: Usar hora local → aplicar duas condições → selecionar.

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [ ]:
for cidade in ["sp","cuiaba"]:
    horas = eventos[cidade].dt.hour
    display(eventos.loc[(horas >= 8) & (horas < 18), ["id",cidade]])

### 4 Reunião e duração
Uma reunião começa às 09h de Cuiabá em 05/10/2026 e dura 90 minutos. Mostre início e fim em UTC e SP e confira a duração.

Antes de programar: quais dados tenho e quais passos preciso seguir?

In [ ]:
# Sua solução aqui.

### Resolução para mostrar na aula

### 4 Reunião e duração

Como pensar: Localizar a hora de origem → somar duração → converter.

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [ ]:
inicio = pd.Timestamp("2026-10-05 09:00:00").tz_localize("America/Cuiaba")
fim = inicio + pd.Timedelta(minutes=90)
for fuso in ["UTC","America/Sao_Paulo","America/Cuiaba"]:
    print(fuso, inicio.tz_convert(fuso), fim.tz_convert(fuso))
assert (fim-inicio).total_seconds()/60 == 90

## Gabarito comentado

### 1 Prazos

Como pensar: Verificar nulos → selecionar durações válidas acima de 4.

In [ ]:
display(pedidos.loc[pedidos["inicio"].isna() | pedidos["fim"].isna()])
display(pedidos.loc[pedidos["dias"] > 4])
assert pedidos.loc[1,"dias"] == 5

### 2 Mudança de dia

Como pensar: Separar válidos → converter → extrair data → contar.

In [ ]:
validos = eventos.loc[eventos["utc"].notna()].copy()
for cidade in ["sp","cuiaba"]:
    display(validos[cidade].dt.date.value_counts().sort_index())
display(eventos.loc[eventos["utc"].isna()])
assert eventos.loc[2,"sp"].day == 5
assert eventos.loc[2,"cuiaba"].day == 4

### 3 Janela comercial

Como pensar: Usar hora local → aplicar duas condições → selecionar.

In [ ]:
for cidade in ["sp","cuiaba"]:
    horas = eventos[cidade].dt.hour
    display(eventos.loc[(horas >= 8) & (horas < 18), ["id",cidade]])

### 4 Reunião e duração

Como pensar: Localizar a hora de origem → somar duração → converter.

In [ ]:
inicio = pd.Timestamp("2026-10-05 09:00:00").tz_localize("America/Cuiaba")
fim = inicio + pd.Timedelta(minutes=90)
for fuso in ["UTC","America/Sao_Paulo","America/Cuiaba"]:
    print(fuso, inicio.tz_convert(fuso), fim.tz_convert(fuso))
assert (fim-inicio).total_seconds()/60 == 90

# Oficina de funções e reutilização

Definir prepara uma função; chamar executa seu corpo. Em notebooks, execute primeiro a célula com def. Faça os exercícios antes de consultar as soluções. As oficinas usam dados próprios e podem ser executadas após os imports.

## Demo Conversão repetida
Origem declarada UTC. Uma função recebe a Series e o fuso de destino; utc=True não deve ser usado para fingir que uma hora local é UTC.

In [ ]:
horarios = pd.Series(["2026-10-05T03:30:00Z","2026-10-05T15:00:00Z","invalido"])
sp_repetido = pd.to_datetime(horarios,format="ISO8601",utc=True,errors="coerce").dt.tz_convert("America/Sao_Paulo")
cuiaba_repetido = pd.to_datetime(horarios,format="ISO8601",utc=True,errors="coerce").dt.tz_convert("America/Cuiaba")

## Pratique a transformação
Para cada atividade, escreva entrada → processamento → saída. Compare o código antes e depois: onde a regra fica definida? Quantos lugares precisam mudar quando ela muda? Funções não tornam todo código mais rápido; o ganho aqui é organização, reutilização e manutenção.

### 1 Extraia a função

Crie converter_utc(textos, destino). Chame para SP e Cuiabá e compare com as duas versões repetidas.

In [ ]:

# Sua solução aqui.

### Resolução para mostrar na aula

### 1 Extraia a função

A interpretação do texto fica em um lugar; o fuso vira parâmetro. Não escondemos NaT.

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [ ]:
def converter_utc(textos,destino):
    instantes = pd.to_datetime(textos,format="ISO8601",utc=True,errors="coerce")
    return instantes.dt.tz_convert(destino)

sp = converter_utc(horarios,"America/Sao_Paulo")
cuiaba = converter_utc(horarios,destino="America/Cuiaba")
assert sp.equals(sp_repetido)
assert cuiaba.equals(cuiaba_repetido)
display(pd.DataFrame({"sp":sp,"cuiaba":cuiaba}))

### 2 Reutilize o filtro

Crie em_expediente(horarios_locais, inicio=8, fim=18) retornando máscara booleana. Aplique em SP e Cuiabá.

In [ ]:

# Sua solução aqui.

### Resolução para mostrar na aula

### 2 Reutilize o filtro

A função recebe horas já convertidas. O início é inclusivo e o fim exclusivo; ausentes não entram.

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [ ]:
def em_expediente(horarios_locais,inicio=8,fim=18):
    horas = horarios_locais.dt.hour
    return (horas >= inicio) & (horas < fim)

print(em_expediente(sp).tolist())
print(em_expediente(cuiaba).tolist())
assert em_expediente(sp).tolist() == [False,True,False]

## Soluções da oficina

### 1 Extraia a função

A interpretação do texto fica em um lugar; o fuso vira parâmetro. Não escondemos NaT.

In [ ]:
def converter_utc(textos,destino):
    instantes = pd.to_datetime(textos,format="ISO8601",utc=True,errors="coerce")
    return instantes.dt.tz_convert(destino)

sp = converter_utc(horarios,"America/Sao_Paulo")
cuiaba = converter_utc(horarios,destino="America/Cuiaba")
assert sp.equals(sp_repetido)
assert cuiaba.equals(cuiaba_repetido)
display(pd.DataFrame({"sp":sp,"cuiaba":cuiaba}))

### 2 Reutilize o filtro

A função recebe horas já convertidas. O início é inclusivo e o fim exclusivo; ausentes não entram.

In [ ]:
def em_expediente(horarios_locais,inicio=8,fim=18):
    horas = horarios_locais.dt.hour
    return (horas >= inicio) & (horas < fim)

print(em_expediente(sp).tolist())
print(em_expediente(cuiaba).tolist())
assert em_expediente(sp).tolist() == [False,True,False]

Referência: https://docs.python.org/3/tutorial/controlflow.html#defining-functions . lambda contém uma expressão; use def quando precisar de várias etapas, validação ou uma regra reutilizada com nome.